<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-04-rag/lesson-4.7-evaluation/notebooks/GCP_Capstone_4.7_Evaluation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.7 Evaluate Retrieval Before You Ship
**Netsetos GenAI Engineering — GCP Capstone** · Module 4 · new in v1.1

You now have three retrieval paths over one corpus and **no evidence** about which is better:

- **vector** — lesson 4.2: dense retrieval, top 5
- **packed** — lesson 4.5: hybrid + Rank API rerank + budgeted packing + cached prefix
- **graph** — lesson 4.6: `retrieval_mode='auto'`, graph for relational questions

This notebook builds the stopwatch: a golden set written as a contract, deterministic retrieval metrics, a model-graded faithfulness gate with a **pinned** judge, a moving baseline, and one comparison table.

It writes an `evals/` directory as it goes; the committed twin is `deploy/evals/` — `golden.jsonl` and `run_eval.py`, the gate 4.8 runs against the live lane and 12.7 runs in CI. Two of the three paths run here on their own, over the lane's `chunks` (vector, packed); the graph path needs 4.6's cells and is skipped until they are loaded. *Facts verified 2026-09-03; the lane's first live run, 2026-09-07.*

## Setup
Two clients again, plus one new rule: **the evaluation service is regional**. It runs in `us-central1`, like embeddings and tuning — not on the `global` endpoint that serves generation.

`JUDGE_MODEL` is pinned here and used everywhere. Step 4 explains why that single line matters more than it looks.

In [ ]:
!pip install -q google-genai==2.21.0 google-cloud-firestore==2.30.0 google-cloud-discoveryengine==0.13.11 \
                 pydantic==2.13.5 pandas==2.3.2 google-cloud-aiplatform==2.1.0
# google-cloud-aiplatform carries the evaluation SDK (vertexai.Client().evals); its module-level imports
# beyond the pins are tqdm and PyYAML, both in Colab's base image - elsewhere: google-cloud-aiplatform[evaluation]
!npm install -g promptfoo@latest        # the eval runner; Node is already on Colab
!promptfoo --version                    # recorded with the run: a different runner is a different run - pin it in your own repo

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"     # CHANGE THIS
TENANT     = "acme"
EVAL_DIR   = "evals"                    # this notebook's workspace; the committed twin is deploy/evals/

import json, os, re, subprocess, time
import pandas as pd
from google import genai
from google.genai import types
import vertexai
from vertexai import types as vtypes   # the evaluation service's own SDK (google-cloud-aiplatform)

# One client for generation (global), one for the EVALUATION service, which is regional: it runs
# in us-central1, like embeddings and tuning - and it is vertexai.Client, not genai.Client:
# the google-genai SDK has no evals surface (checked against 2.21.0).
gen_client  = genai.Client(enterprise=True, project=PROJECT_ID, location="global")
eval_client = vertexai.Client(project=PROJECT_ID, location="us-central1")

ANSWER_MODEL = "gemini-3.6-flash"
JUDGE_MODEL  = "gemini-3.1-flash-lite"   # PINNED. Never inherit a default autorater - see Step 6.

os.makedirs(EVAL_DIR, exist_ok=True)
print("eval workspace:", os.path.abspath(EVAL_DIR))

## Cell 1: The golden set — a contract, not a question
A question alone is not scorable; you still have to decide by hand whether the answer was right. A contract says up front what you will accept, so a machine can decide.

Four shapes, because they fail for different reasons and are fixed by different work:

| shape | tests | when it fails, look at |
|---|---|---|
| lookup | one chunk holds the answer | embedding, chunk size, distance threshold |
| join | the answer needs two clauses | packing order and the token budget (4.5) |
| refusal | the corpus has no answer | the system prompt and the grounding rule |
| isolation | asked as a different tenant | nothing — this is a leak, not a score |
| version | the current version of a document answers; a retired one is never cited | the ledger (12.5): the swap, the retirement — or the rows that did not move |

> The **isolation** row is different in kind. There is no threshold at which a data leak is acceptable, so it is a boolean and it blocks a release on its own.

A fifth shape arrived with the ledger (12 September 2026, `deploy/INDEXING.md`): a **version** row names its `source` document, expects the current version's figure and forbids the figure a retired version under `evals/demo` carries. The falsifiability check below is what forces the rows to move with the corpus: re-issue the handbook without moving `lk-06` and `vr-01`, and the offline gate is red before anything is deployed — a document change goes through the same gates as a code change.

Since the corpus holds real documents — thirteen under ACME, the four Labour Codes under Zeta, the DPDP and IT Acts under Globex — the same four shapes cover them: a lookup asserts the statute's own words (`twenty-six weeks`, not `26 weeks`), a join spans the amending Act and the principal Act, and an isolation row asks Zeta a question only ACME's documents can answer.

In [ ]:
# A golden row is a CONTRACT, not a question. Each one names the answer you will accept and
# the chunks that must be retrieved for the answer to be honest.
#
# These eighteen are copied from deploy/evals/golden.jsonl - same ids, same wording - so what
# you score here is what CI scores in 12.7. Every must_contain is a string the corpus holds:
# the handbook's figures, and for the real Acts the statute's OWN words ("twenty-six weeks",
# "five years"), which a model answering from memory phrases differently. The isolation rows
# carry must_not_contain: the other tenant's figure that must never appear.
GOLDEN = [
    # --- lookup: one chunk holds the answer (the shape 4.2 already handles) ---
    {"id": "lk-01", "shape": "lookup", "question": "What is the per-trip cap on domestic travel reimbursement?",
     "tenant": "acme", "must_contain": ["40,000"], "must_retrieve": ["EXP-12", "hr_policy_2026"], "answerable": True},
    {"id": "lk-02", "shape": "lookup", "question": "By when is Form 16 issued?",
     "tenant": "acme", "must_contain": ["15 June"], "must_retrieve": ["PR-05", "hr_policy_2026"], "answerable": True},
    {"id": "lk-03", "shape": "lookup", "question": "How many days of earned leave can I carry forward?",
     "tenant": "acme", "must_contain": ["30"], "must_retrieve": ["LV-01", "hr_policy_2026"], "answerable": True},
    {"id": "lk-04", "shape": "lookup", "question": "What notice period applies during probation?",
     "tenant": "acme", "must_contain": ["15"], "must_retrieve": ["PB-02", "hr_policy_2026"], "answerable": True},
    {"id": "lk-05", "shape": "lookup", "question": "Are USB drives allowed on a company laptop?",
     "tenant": "acme", "must_contain": ["blocked"], "must_retrieve": ["IT-SEC-04", "hr_policy_2026"], "answerable": True},

    # --- lookup over a REAL document: the statute's own wording, on its own page ---
    {"id": "lk-14", "shape": "lookup", "question": "What is the maximum period of maternity benefit after the 2017 amendment?",
     "tenant": "acme", "must_contain": ["twenty-six weeks"], "must_retrieve": ["maternity_benefit_amendment_act_2017"], "answerable": True},
    {"id": "lk-16", "shape": "lookup", "question": "After how many years of continuous service does gratuity become payable?",
     "tenant": "acme", "must_contain": ["five years"], "must_retrieve": ["payment_of_gratuity_act_1972"], "answerable": True},
    {"id": "lk-26", "shape": "lookup", "question": "What is a Consent Manager under the DPDP Act?",
     "tenant": "acme", "must_contain": ["single point of contact"], "must_retrieve": ["dpdp_act_2023"], "answerable": True},

    # --- join: the answer needs two clauses, so packing order and budget matter (4.5) ---
    {"id": "jn-01", "shape": "join", "question": "If I resign during probation, what notice applies and can I encash leave?",
     "tenant": "acme", "must_contain": ["15", "cannot"], "must_retrieve": ["PB-02", "LV-07"], "answerable": True},
    {"id": "jn-02", "shape": "join", "question": "Can unused leave shorten my notice period?",
     "tenant": "acme", "must_contain": ["cannot"], "must_retrieve": ["NP-03", "LV-07"], "answerable": True},

    # --- join across two REAL documents: the amending Act and the principal Act; the 1972 Act and the 2020 Code ---
    {"id": "jn-08", "shape": "join", "question": "The principal Maternity Benefit Act fixes the maximum period at twelve weeks. What did the 2017 amendment change it to?",
     "tenant": "acme", "must_contain": ["twelve weeks", "twenty-six weeks"], "must_retrieve": ["maternity_benefit_act_1961", "maternity_benefit_amendment_act_2017"], "answerable": True, "note": "A real two-document join: the new figure is in the amending Act, the section it replaces is in the principal Act, and neither document alone answers the question."},
    {"id": "jn-10", "shape": "join", "question": "The Payment of Gratuity Act requires five years of continuous service. What does the Code on Social Security say for a fixed term employee?",
     "tenant": "acme", "must_contain": ["five years", "pro rata basis"], "must_retrieve": ["payment_of_gratuity_act_1972", "code_on_social_security_2020"], "answerable": True, "note": "A real change in the law across two real documents: the 1972 Act's five years and the 2020 Code's pro rata gratuity for fixed term employment. Neither document alone answers it."},

    # --- refusal: the corpus does NOT contain this. The right answer is to say so. ---
    {"id": "rf-01", "shape": "refusal", "question": "What is ACME's sabbatical policy?",
     "tenant": "acme", "must_contain": [], "must_retrieve": [], "answerable": False, "note": "Was 'parental leave' until the real Maternity Benefit Acts joined ACME's corpus, which made twenty-six weeks a defensible answer. A refusal row has to ask for something the corpus really does not hold."},
    {"id": "rf-02", "shape": "refusal", "question": "How many casual leave days do I get?",
     "tenant": "acme", "must_contain": [], "must_retrieve": [], "answerable": False},

    # --- the rows that must never go green: tenant isolation ---
    {"id": "iso-01", "shape": "isolation", "question": "What is the per-trip cap on travel reimbursement?",
     "tenant": "zeta", "must_contain": ["25,000"], "must_retrieve": ["EXP-12", "hr_policy_zeta_2026"], "answerable": True, "must_not_contain": ["40,000"], "note": "The sharpest shape: the SAME question, a different tenant. Zeta's own cap is Rs 25,000 and the row is answerable - what must never appear is ACME's Rs 40,000."},
    {"id": "iso-06", "shape": "isolation", "question": "What is the maximum period of maternity benefit?",
     "tenant": "globex", "must_contain": [], "must_retrieve": [], "answerable": False, "must_not_contain": ["twenty-six weeks"], "note": "Globex holds no maternity law. Twenty-six weeks can only have come from ACME's Amendment Act or Zeta's Code on Social Security. (Asked of Zeta until the Code arrived - the Code consolidates the Maternity Benefit Act, so Zeta now answers it: lk-31.)"},
    {"id": "iso-08", "shape": "isolation", "question": "What is the maximum rate of central tax the CGST Act allows?",
     "tenant": "zeta", "must_contain": [], "must_retrieve": [], "answerable": False, "must_not_contain": ["recommendations of the Council"], "note": "Only ACME holds the CGST Act. 'Twenty per cent' also lives in Zeta's Code on Wages (the bonus ceiling), so the leak marker is the Act's own phrase."},

    # --- the ledger's row (12 September 2026): the current version answers, a retired one is never cited ---
    {"id": "vr-01", "shape": "version", "question": "What is the notice period for a confirmed E3?",
     "tenant": "acme", "must_contain": ["60"], "must_retrieve": ["NP-03", "hr_policy_2026"], "answerable": True,
     "must_not_contain": ["90 days"], "source": "hr_policy_2026.md",
     "note": "lk-06 asks the same question; this row adds what must NOT be said: revision 2 of the handbook (evals/demo/hr_policy_2026_v2.md) makes it 90 days. When the handbook is re-issued, this row and lk-06 move to 90 in the same commit as the corpus - the red gate in between is the demo."},
]
print(f"{len(GOLDEN)} golden rows: " +
      ", ".join(f"{s}={sum(1 for r in GOLDEN if r['shape'] == s)}"
                for s in ("lookup", "join", "refusal", "isolation", "version")))


### Write it as JSON Lines
One row per line means a change to the contract is a readable diff in code review. When someone loosens a `must_contain` to make a test pass, that is a line change with an author and a date. In a spreadsheet it is invisible.

In [ ]:
GOLDEN_PATH = os.path.join(EVAL_DIR, "golden.jsonl")

def write_jsonl(path: str, rows: list) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {len(rows)} rows -> {path}")

# The committed set is a SUPERSET of the rows above: 65 rows, eleven of them isolation and
# twenty-seven over the real documents (deploy/evals/build_golden.py verifies every figure
# against the corpus before it writes). When the repo is beside this notebook, score that one.
KIT_GOLDEN = next((p for p in ("deploy/evals/golden.jsonl", "../deploy/evals/golden.jsonl",
                               "/content/agentic-ai-weekend-gcp-learners/deploy/evals/golden.jsonl") if os.path.exists(p)), None)
if KIT_GOLDEN:
    GOLDEN = [json.loads(l) for l in open(KIT_GOLDEN, encoding="utf-8") if l.strip()]
    print(f"using the committed golden set: {len(GOLDEN)} rows from {KIT_GOLDEN}")
write_jsonl(GOLDEN_PATH, GOLDEN)

# One row per line, so a diff on this file is a readable review of a change to the contract.
# Lesson 12.7 runs deploy/evals/golden.jsonl in CI on every push (run_eval.py).
print(open(GOLDEN_PATH, encoding="utf-8").readline().strip()[:160], "...")


### Where the corpus comes from
The rows are checked against a corpus the kit builds, not one it collected: `deploy/evals/build_corpus.py` for the synthetic tenants (format-valid, invented PII), `fetch_real.py` for the statutes. Read from the clone when it is beside the notebook.


In [ ]:
# The corpus the rows above are verified against is BUILT, and the builder sits in the kit beside the golden set.
# deploy/evals/build_corpus.py writes the three synthetic tenants - ACME, Zeta and Globex: contracts, a Hinglish
# invoice, a 40k-token policy pack - with every PAN, GSTIN and Aadhaar format-valid and invented, so the DLP demos
# fire and nothing real is ever on a screen; fetch_real.py puts the real statutes beside them. Deterministic: the
# same input gives the same bytes, so a diff under corpus/ is a change somebody made on purpose. The clause ids the
# golden rows cite (EXP-12, LV-01, PB-02, NP-03, LV-07, PR-05, IT-SEC-04) are minted there, which is why
# build_golden.py can check every figure against the corpus before it writes a row.
if KIT_GOLDEN:
    EVALS_DIR = os.path.dirname(KIT_GOLDEN)
    src = open(os.path.join(EVALS_DIR, "build_corpus.py"), encoding="utf-8").read()
    print("\n".join(l for l in src.splitlines() if l.startswith(("TENANTS", "def "))))
    manifest = json.load(open(os.path.join(EVALS_DIR, "manifest.json"), encoding="utf-8"))
    print(f"\nmanifest.json: {len(manifest)} corpus files, {sum(1 for m in manifest if m.get('approx_tokens'))} of them text; "
          f"{sum(1 for m in manifest if m['tenant_id'] == TENANT)} under {TENANT}")
else:
    print("no clone beside this notebook: git clone -b main https://github.com/netsetos/agentic-ai-weekend-gcp-learners")


## Cell 2: Measure what you can count before you pay a judge
`must_retrieve` already told you the right answer, so two of the most useful numbers in retrieval evaluation need no model at all.

- **recall@k** — did the required chunks reach the top k? Separates a *retrieval* failure from a *generation* failure. If recall is 0, no prompt change will save the answer.
- **MRR** — how high was the first required chunk? A chunk at position 20 gets dropped by 4.5's packer.

Both are deterministic, so a change means your retriever changed — never that a grader model had a different morning. Both are free, so run them on every commit.

In [ ]:
def _hay(c: dict) -> str:
    """What an anchor is matched against: the chunk's source file and its text, as deploy/evals/
    run_eval.py does. The lane's chunk ids (acme:<sha256>#<i>) name neither the clause nor the
    document, so a match on ids alone scores every row 0.0 whatever was retrieved."""
    return (c.get("source_uri", "") + " " + c.get("text", "")).lower()

def recall_at_k(chunks: list, must_retrieve: list, k: int = 5) -> float:
    """Fraction of the required anchors that appear in the top k. No judge, no cost, no drift."""
    if not must_retrieve:
        return 1.0                                  # nothing required: trivially satisfied
    hay = [_hay(c) for c in chunks[:k]]
    return sum(1 for want in must_retrieve if any(want.lower() in h for h in hay)) / len(must_retrieve)

def mrr(chunks: list, must_retrieve: list) -> float:
    """Reciprocal rank of the FIRST required anchor. Rewards putting it first, not just in the list."""
    if not must_retrieve:
        return 1.0
    for rank, c in enumerate(chunks, 1):
        if any(want.lower() in _hay(c) for want in must_retrieve):
            return 1.0 / rank
    return 0.0

# Measure what you can count before you pay a judge. These two numbers are deterministic:
# the same run gives the same answer, so a change in them is a change in your retriever -
# never a mood swing in a grader model.
demo = [{"source_uri": "gs://documind-ai-YOUR-ID-uploads/acme/hr_policy_2026.md", "text": "PB-02 - Probation: fifteen days notice either side"},
        {"source_uri": "gs://documind-ai-YOUR-ID-uploads/acme/hr_policy_2026.md", "text": "LV-07 - Encashment: at basic pay on exit"},
        {"source_uri": "gs://documind-ai-YOUR-ID-uploads/acme/hr_policy_2026.md", "text": "EXP-12 - Domestic travel: Rs 40,000 a trip"}]
print("recall@5:", recall_at_k(demo, ["PB-02", "LV-07"]))
print("mrr     :", round(mrr(demo, ["LV-07"]), 3), "(second position -> 0.5)")


In [ ]:
_SMALL = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
          "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13,
          "fourteen": 14, "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
          "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
          "seventy": 70, "eighty": 80, "ninety": 90}
_SCALE = {"hundred": 100, "thousand": 1000, "lakh": 100000, "crore": 10000000}
_WORD = "|".join(list(_SMALL) + list(_SCALE))
_NUMWORDS = re.compile(r"\b(?:(?:%s)(?:[\s-]+(?:%s))*)\b" % (_WORD, _WORD))

def _to_int(run: str) -> int:
    total = current = 0
    for w in re.split(r"[\s-]+", run):
        if w in _SMALL:
            current += _SMALL[w]
        elif w == "hundred":
            current = (current or 1) * 100
        elif w in _SCALE:
            total += (current or 1) * _SCALE[w]
            current = 0
    return total + current

def normalise(text: str) -> str:
    """Lower-case, thousands separators out, number words to digits - on BOTH sides of a containment
    check, so "twelve weeks" and "12 weeks" are the same fact. The live gate (deploy/evals/run_eval.py)
    does exactly this since 7 Sept 2026: a right figure in the statute's own spelling was scored wrong."""
    t = text.lower().replace(",", "")
    return _NUMWORDS.sub(lambda m: str(_to_int(m.group(0))), t)


## Cell 3: Wire up the three paths
Two are wired here, over the lane's `chunks`: **vector** (4.2's dense top 5) and **packed** (4.5's shape: dense 20, Rank API 5, a token budget). **graph** needs 4.6's `GRAPH`, `retrieve()` and `answer_with_graph()` loaded in this runtime; until then it is skipped and the table has two columns.

> The unwired stub **raises** rather than returning nothing on purpose. An unwired path that returned an empty answer would score as a refusal, and refusals count as correct on the unanswerable rows — so a completely broken path could post a respectable score. A crash inside a wired path is recorded as a failed row for the same reason.

### The answer contract
Every path returns a `RAGAnswer` — the one shape 3.2 defined and 4.2, 4.5, 4.6 and the rag-api carry — so the scorer reads `answerable` and the citations from the contract instead of hunting for refusal phrases in prose. Pasted verbatim from `deploy/shared/documind_schemas.py`, with the `draft_of()` repair from 4.5.

In [ ]:
# THE answer contract - copied verbatim from deploy/shared/documind_schemas.py at build time,
# the same text 3.2, 4.2, 4.5, 4.6 and the rag-api service carry. The scorer below reads
# `answerable` and the citations from this shape: a refusal is a field, never a keyword hunt.
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, ValidationError

class Citation(BaseModel):
    """One passage a caller can open. Text by default; a figure, a table or a video segment
    when the chunk is one - the four extra fields are optional and additive, so every citation
    written before Module 9 comes through unchanged."""

    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)
    kind: Literal["text", "figure", "table", "segment"] = "text"
    media_url: Optional[str] = None            # a signed URL for figure / table / segment
    start: Optional[float] = None              # seconds, for a video or audio segment
    end: Optional[float] = None


class RAGAnswer(BaseModel):
    """The contract. Modules 3 to 13 all pass this shape along."""

    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool


class DraftCitation(BaseModel):
    """What the model cites: the [Source N] number it saw, and the words it is relying on."""

    source: int = Field(ge=1, description="1-based [Source N] in the context")
    quote: str = Field(max_length=200, description="Exact words from that source")


class ModelDraft(BaseModel):
    """What the model is asked for. Use this as response_schema; resolve() turns it into a
    RAGAnswer. Asking the model for chunk ids or scores directly invites it to invent them."""

    answer: str
    citations: List[DraftCitation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool

def resolve(draft: ModelDraft, packed: List[dict]) -> RAGAnswer:
    """Turn the model's [Source N] citations into Citations, against the chunks it SAW.

    `packed` must be the list the context was built from - after the token budget dropped
    anything - not the list retrieval returned. Indexing into the pre-budget list shifts every
    citation after a dropped chunk by one, silently, which is exactly the bug rag-api had.

    An out-of-range index is dropped, not raised: the model's answer is still worth returning,
    and a citation to a source that was not in the context is not a citation.
    """
    cits: List[Citation] = []
    for d in draft.citations:
        if not 1 <= d.source <= len(packed):
            continue
        c = packed[d.source - 1]
        cits.append(Citation(
            chunk_id=str(c.get("chunk_id") or c.get("id") or ""),
            source_uri=c.get("source_uri", ""),
            page=c.get("page_start") or c.get("page"),
            quote=(d.quote or c.get("text", ""))[:500],
            score=float(min(1.0, max(0.0, c.get("rerank_score") or c.get("score") or 0.0))),
            kind=c.get("kind", "text"),
            media_url=c.get("media_url"),
            start=c.get("start"),
            end=c.get("end"),
        ))
    return RAGAnswer(answer=draft.answer, citations=cits,
                     confidence=draft.confidence, answerable=draft.answerable)

import json
from pydantic import ValidationError

def draft_of(r, schema, quote_limit=200):
    """The model's structured answer, or a clear error - never a silent None.

    The SDK sets r.parsed to None on ANY validation failure. The first live run of the lane
    (7 Sept 2026) met the one that matters: a quote longer than the contract's 200 characters -
    a statute provision is one sentence - which threw fourteen right answers away and, worse,
    had been scored as the model refusing. So: read the JSON, trim the quote to the contract,
    validate again; anything else is an error you can read, not a refusal."""
    if r.parsed is not None:
        return r.parsed if isinstance(r.parsed, schema) else schema.model_validate(r.parsed)
    cand = (r.candidates or [None])[0]
    reason = getattr(getattr(cand, "finish_reason", None), "name", "NO_CANDIDATES")
    try:
        obj = json.loads(r.text or "")
    except ValueError:
        raise RuntimeError(f"no JSON to parse (finish_reason={reason}) - raise max_output_tokens if it is MAX_TOKENS")
    if isinstance(obj, dict):
        for c in obj.get("citations") or []:
            if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > quote_limit:
                c["quote"] = c["quote"][: quote_limit - 3].rstrip() + "..."
    try:
        return schema.model_validate(obj)
    except ValidationError as e:
        err = e.errors()[0]
        raise RuntimeError(f"the draft failed the contract at {'.'.join(str(x) for x in err['loc'])}: {err['msg']}") from None


In [ ]:
# The two paths this notebook runs on its own, over the lane's chunks: 4.2's dense top-5, and
# 4.5's rerank-and-pack (dense 20 -> Rank API 5 -> a token budget; the BM25 half and the cache
# stay in 4.5). Each returns (RAGAnswer, chunks). The graph path needs 4.6's cells - GRAPH,
# retrieve(), answer_with_graph() - so it raises here, and a path that raises NotImplementedError
# is SKIPPED loudly, never scored: an unwired path that returned nothing would score as a refusal,
# and refusals count as correct on the unanswerable rows.
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud import discoveryengine_v1 as discoveryengine

emb_client = genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")   # embeddings: regional
db = firestore.Client(project=PROJECT_ID)
_ranker = discoveryengine.RankServiceClient()
_ranking_config = _ranker.ranking_config_path(project=PROJECT_ID, location="global", ranking_config="default_ranking_config")

# The lane's own system prompt (deploy/services/rag-api/generator.py), verbatim - the answers
# scored here are produced the way the API produces them.
SYSTEM = """You are DocuMind, a retrieval-grounded assistant.
Rules:
1. Answer ONLY from the numbered context below. Never invent sources.
2. Cite using [N] where N is the chunk number. Multiple chunks: [1,2].
3. If the context does not contain the answer, set answerable=false and say so.
4. Keep answers under 300 words unless asked for more.
5. A quote is the clause that answers - at most twenty-five words, never a whole section.
"""

def embed_query(q: str) -> list:
    return emb_client.models.embed_content(
        model="text-embedding-005", contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768)).embeddings[0].values

def dense(question: str, tenant: str, k: int) -> list:
    """Firestore find_nearest with the tenant PRE-filter - 4.2's retriever."""
    docs = (db.collection("chunks").where(filter=FieldFilter("tenant_id", "==", tenant))
              .find_nearest(vector_field="embedding", query_vector=Vector(embed_query(question)),
                            distance_measure=DistanceMeasure.COSINE, limit=k, distance_result_field="vector_distance")
              .get())
    out = []
    for d in docs:
        x = d.to_dict()
        out.append({"chunk_id": d.id, "text": x.get("text", ""), "source_uri": x.get("source_uri", ""),
                    "page_start": x.get("page_start"), "score": round(1 - x.get("vector_distance", 1.0), 4)})
    return out

def rerank(question: str, chunks: list, top_n: int = 5, model: str = "semantic-ranker-fast-004") -> list:
    """The Rank API, served from global - 4.5's cross-encoder step."""
    if not chunks:
        return chunks
    records = [discoveryengine.RankingRecord(id=str(i), title=c["source_uri"].rsplit("/", 1)[-1], content=c["text"][:4000])
               for i, c in enumerate(chunks)]
    resp = _ranker.rank(request=discoveryengine.RankRequest(
        ranking_config=_ranking_config, model=model, top_n=top_n, query=question, records=records))
    out = []
    for r in resp.records:
        c = dict(chunks[int(r.id)]); c["rerank_score"] = round(r.score, 3); out.append(c)
    return out

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def block(n: int, c: dict) -> str:
    page = f", p.{c['page_start']}" if c.get("page_start") else ""
    return f"[Source {n}] ({c['source_uri'].rsplit('/', 1)[-1]}{page})\n{c['text']}"

def pack(chunks: list, budget: int):
    """4.5's packer: most-relevant-first, a chunk that does not fit is dropped, the rest are tried."""
    packed, parts, used = [], [], 0
    for c in chunks:
        b = block(len(packed) + 1, c); need = estimate_tokens(b)
        if used + need > budget:
            continue
        used += need; packed.append(c); parts.append(b)
    return "\n\n".join(parts), packed

def small_budget(chunks: list) -> int:
    """Two of the top chunks fit, the third does not - whatever their size (4.5's deliberate failure,
    stated in chunks: the lane's windows are ~500 tokens, the loader's clauses ~80)."""
    needs = [estimate_tokens(block(i, c)) for i, c in enumerate(chunks, 1)]
    return sum(needs[:2]) + (needs[2] // 2 if len(needs) > 2 else 0)

def generate(question: str, context: str, packed: list) -> RAGAnswer:
    r = gen_client.models.generate_content(
        model=ANSWER_MODEL, contents=f"Context:\n{context}\n\nQuestion: {question}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM, response_mime_type="application/json", response_schema=ModelDraft,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"), max_output_tokens=2048))
    return resolve(draft_of(r, ModelDraft), packed)   # against the packed list: a dropped chunk shifts every later [Source N]

def path_vector(question: str, tenant: str = TENANT):
    """Lesson 4.2: dense retrieval, top 5, no reranking, no budget."""
    chunks = dense(question, tenant, 5)
    context = "\n\n".join(block(i, c) for i, c in enumerate(chunks, 1))
    return generate(question, context, chunks), chunks

def path_packed(question: str, tenant: str = TENANT, chunk_budget=6_000):
    """Lesson 4.5's shape: dense 20 -> Rank API 5 -> packed to a budget ("small" = two of the five)."""
    top = rerank(question, dense(question, tenant, 20), top_n=5)
    budget = small_budget(top) if chunk_budget == "small" else chunk_budget
    context, packed = pack(top, budget)
    if not packed:
        return RAGAnswer(answer="No relevant context.", citations=[], confidence="low", answerable=False), []
    return generate(question, context, packed), packed

def path_graph(question: str, tenant: str = TENANT):
    """Lesson 4.6: retrieval_mode='auto' - graph when the question is relational."""
    raise NotImplementedError("needs 4.6's cells (GRAPH, retrieve, answer_with_graph) in this runtime")

PATHS = {"vector": path_vector, "packed": path_packed, "graph": path_graph}

def run_path(name: str, rows: list) -> pd.DataFrame:
    """Run one path over the golden set. Records the answer, the chunks and the metrics."""
    fn, out = PATHS[name], []
    for r in rows:
        tenant = r.get("tenant", TENANT)
        try:
            answer, chunks = fn(r["question"], tenant)
        except NotImplementedError:
            raise                                    # not wired: the caller skips the path, loudly
        except Exception as e:                       # a crash is a FAILED row, not a missing row
            answer, chunks = RAGAnswer(answer=f"ERROR: {e}", citations=[], confidence="low", answerable=False), []
        text = normalise(answer.answer)
        out.append({
            "id": r["id"], "shape": r["shape"], "path": name, "question": r["question"], "tenant": tenant,
            "answer": answer.answer, "answerable": answer.answerable, "cites": len(answer.citations),
            "chunk_ids": ",".join(c["chunk_id"] for c in chunks),
            "context": "\n\n".join(c["text"] for c in chunks),            # what Cell 4 grades faithfulness against
            "recall@5": recall_at_k(chunks, r["must_retrieve"]),
            "mrr": mrr(chunks, r["must_retrieve"]),
            "contains_all": all(normalise(s) in text for s in r["must_contain"]),      # the figure, not the spelling (4.8, F22)
            "leak": any(normalise(s) in text for s in r.get("must_not_contain", [])) if r["shape"] != "version" else False,   # the other tenant's figure
            "stale": any(normalise(s) in text for s in r.get("must_not_contain", [])) if r["shape"] == "version" else False,  # a retired version's figure (the ledger)
            "refused": not answer.answerable,                                           # the contract's field, not a keyword
            "expect_answer": r["answerable"],
        })
        time.sleep(0.1)
    return pd.DataFrame(out)


### Score a run
Four numbers and one boolean per path. `answer_match` and `refusal_rate` are deliberately separate: a system that answers everything scores well on the first and badly on the second, and that is the trade you most need to see. Over-answering is the dangerous direction.

In [ ]:
def score(df: pd.DataFrame) -> dict:
    """Turn a run into the numbers a release decision needs - the live gate's, in the same words."""
    answerable = df[df["expect_answer"]]
    unanswerable = df[~df["expect_answer"]]
    answered = answerable[answerable["answerable"]]
    iso = df[df["shape"] == "isolation"]
    return {
        "rows": len(df),
        "recall@5": round(answerable["recall@5"].mean(), 3) if len(answerable) else None,
        "mrr": round(answerable["mrr"].mean(), 3) if len(answerable) else None,
        # did it say the thing the contract requires?
        "answer_match": round(answerable["contains_all"].mean(), 3) if len(answerable) else None,
        # did an answer come with a citation? (an uncited answer is an opinion)
        "citation_rate": round((answered["cites"] > 0).mean(), 3) if len(answered) else None,
        # did it refuse when it should have? (over-answering is the dangerous direction)
        "refusal_rate": round(unanswerable["refused"].mean(), 3) if len(unanswerable) else None,
        # the blocker: the other tenant's figure in an answer. A yes or no, not a score.
        "isolation_clean": bool(not iso["leak"].any()) if len(iso) else None,
        # the ledger's promise (12.5): a retired version's figure never cited. Also a yes or no.
        "version_clean": bool(not df[df["shape"] == "version"]["stale"].any()) if (df["shape"] == "version").any() else None,
    }

def compare(frames: dict) -> pd.DataFrame:
    """vector vs packed vs graph, side by side. This table is the point of the lesson."""
    return pd.DataFrame({name: score(df) for name, df in frames.items()}).T


### Run all three
Each path over the whole golden set. Keep the frames: the promptfoo tests and the comparison table both read from them.

> Each frame carries a `context` column — the joined chunk text that path produced. Cell 4 grades faithfulness against it, and only against it.

In [ ]:
FRAMES = {}
for name in PATHS:
    try:
        FRAMES[name] = run_path(name, GOLDEN)
    except NotImplementedError as e:
        print(f"{name}: not wired in this runtime ({e}) - skipped, not scored")

for name, df in FRAMES.items():
    print(name, score(df))

retrieved_packed = FRAMES["packed"]          # the frame the faithfulness gate reads


## Cell 4: The faithfulness gate — and the pinned judge
Deterministic metrics cannot answer the question that matters most: *is this answer actually supported by the context it was given?* That needs a reader. `context-faithfulness` hands the answer and the retrieved context to a judge and asks exactly that.

> **Pin the judge.** Left unset, the tool picks its own default — and published examples still name a Gemini 2.0 model, which retires. Three things then go wrong: your scores move when someone else ships a new default and you cannot tell that from a regression; the gate eventually fails on a model that no longer exists; and your historical baseline was produced by a different grader, so comparing across time is meaningless. Changing the judge should be a deliberate event that resets the baseline.

In [ ]:
CONFIG_PATH = os.path.join(EVAL_DIR, "promptfooconfig.yaml")

# 12 September 2026: the judge reads the answer each path ACTUALLY gave. The tests carry it (`answer`, from the
# frame), the `echo` provider returns its prompt as the output, so the output IS the recorded answer, and
# context-faithfulness grades that answer against the context that path retrieved. Before, promptfoo generated a
# fresh answer behind a different prompt - no system rules, no schema - and graded that: a run could pass or fail
# on an answer the pipeline never produced.
CONFIG = f"""# DocuMind retrieval eval, generated and run by lesson 4.7. It is this notebook's runner:
# 12.7's CI gate is deploy/evals/run_eval.py over the same golden set, with no model.
description: DocuMind answer faithfulness on the ACME golden set

prompts:
  - "{{{{answer}}}}"

providers:
  - echo

defaultTest:
  options:
    # PIN THE JUDGE. Left unset, promptfoo picks its own default grader - and the published
    # examples still name a Gemini 2.0 model, which retires. An unpinned judge means your
    # scores move when someone else ships, and you cannot tell that from a regression.
    provider:
      id: vertex:{JUDGE_MODEL}
      config:
        projectId: {PROJECT_ID}
        region: global
  assert:
    - type: context-faithfulness
      threshold: 0.8

tests: file://promptfoo_tests.jsonl
"""
open(CONFIG_PATH, "w", encoding="utf-8").write(CONFIG)
print(CONFIG)

### The test file
One promptfoo test per golden row, carrying the context **that path actually retrieved**.

> Grading against the whole corpus instead would pass an answer that is true but quoted a chunk the retriever never surfaced — precisely the hallucination you are trying to catch.

The unanswerable rows swap the assertion list entirely: checking a refusal by keyword is brittle, because there are many ways to decline, so a rubric asks whether the response declined *and* avoided inventing a figure.

In [ ]:
TESTS_PATH = os.path.join(EVAL_DIR, "promptfoo_tests.jsonl")

def promptfoo_tests(rows: list, retrieved: pd.DataFrame) -> list:
    """One promptfoo test per golden row, carrying the CONTEXT that path actually retrieved.

    context-faithfulness grades the answer against this context, so the context must be the real
    retrieved text - not the whole corpus. Grading against everything would pass an answer that
    quoted a chunk the retriever never returned.
    """
    by_id = {r["id"]: r for r in rows}
    tests = []
    for _, row in retrieved.iterrows():
        g = by_id[row["id"]]
        asserts = [{"type": "context-faithfulness", "threshold": 0.8}]
        for s in g["must_contain"]:
            asserts.append({"type": "icontains", "value": s})
        if not g["answerable"]:
            # the refusal contract, checked by a rubric rather than a keyword
            asserts = [{"type": "llm-rubric",
                        "value": "The response declines to answer and does not state a specific "
                                 "figure, date or policy code."}]
        tests.append({"description": f'{g["id"]} ({g["shape"]})',
                      "vars": {"question": g["question"], "context": row["context"], "answer": row["answer"]},
                      "assert": asserts})
    return tests

# retrieved_packed must carry a 'context' column - the joined chunk text that path produced.
write_jsonl(TESTS_PATH, promptfoo_tests(GOLDEN, retrieved_packed))

### Run the gate
The non-zero exit code is what turns a report into a gate. A report gets read when someone remembers; a non-zero exit stops the pipeline.

In [ ]:
# promptfoo exits non-zero when an assertion fails, which is what makes it a gate rather than a
# report. The lane's gate (deploy/evals/run_eval.py, 4.8 and 12.7) exits the same way.
r = subprocess.run(
    ["promptfoo", "eval", "-c", "promptfooconfig.yaml", "--no-progress-bar"],
    cwd=EVAL_DIR, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("\nGATE FAILED (exit", r.returncode, ") - this is the point. Read the failing rows above.")
else:
    print("\nGATE PASSED")

## Cell 5: A second opinion, and a baseline that moves
A **floor** is absolute: never ship below 0.85. A **moving baseline** is relative: never ship materially worse than last time. You need both — a floor alone permits a long slide from 0.97 to 0.86 without ever tripping, and a baseline alone lets a system that was always bad stay bad.

> Only a **passing** run may move the baseline. If a failing run wrote its score, one bad release would lower the bar permanently and the next regression would look like an improvement. The 0.02 tolerance is the judge's noise floor.

The call is wrapped: the evaluation client surface and the metric catalogue are both moving, so a rename prints a message rather than killing the notebook. **promptfoo above is the gate that runs in CI**; this is the cross-check on the gate.

In [ ]:
BASELINE_PATH = os.path.join(EVAL_DIR, "baseline.json")
GROUNDEDNESS_FLOOR = 0.85

# The judge's brief, as a metric of the managed Gen AI evaluation service (regional). The
# template's {context} is the column of the same name; the judge is PINNED, like promptfoo's.
GROUNDED = vtypes.LLMMetric(
    name="groundedness",
    prompt_template=(
        "You are grading whether an answer is supported by the context it was given.\n\n"
        "Context:\n{context}\n\nQuestion: {prompt}\n\nAnswer: {response}\n\n"
        "Score 1 if every factual claim in the answer appears in the context, 0 otherwise. "
        "Answer with the score on the first line and one sentence of reasoning on the second."),
    judge_model=JUDGE_MODEL)

def vertex_groundedness(retrieved: pd.DataFrame) -> float | None:
    """Second opinion from the managed Gen AI Evaluation service (regional: us-central1).

    The client surface and the metric catalogue are both moving, so this is wrapped: a rename
    prints a message and returns None instead of failing the notebook. promptfoo above is the
    gate this notebook runs; this is the cross-check on the gate itself.
    """
    df = pd.DataFrame({"prompt": retrieved["question"],
                       "response": retrieved["answer"],
                       "context": retrieved["context"]})
    try:
        result = eval_client.evals.evaluate(dataset=df, metrics=[GROUNDED])
        result.show()
        mean = next((m.mean_score for m in (result.summary_metrics or [])
                     if (m.metric_name or "").startswith("groundedness") and m.mean_score is not None), None)
        if mean is None:
            raise RuntimeError("no groundedness mean in summary_metrics")
        return float(mean)
    except Exception as e:
        print("Vertex evaluation unavailable or the metric surface changed:", str(e)[:200])
        print("Verify the metric and client surface on the Gen AI evaluation docs, "
              "then re-run. The promptfoo gate above is unaffected.")
        return None

def check_against_baseline(name: str, groundedness: float | None) -> bool:
    """A MOVING baseline: today's score must clear the floor AND not regress on the last run."""
    if groundedness is None:
        print("no score - baseline unchanged"); return True
    base = json.load(open(BASELINE_PATH)) if os.path.exists(BASELINE_PATH) else {}
    prev = base.get(name)
    ok = groundedness >= GROUNDEDNESS_FLOOR and (prev is None or groundedness >= prev - 0.02)
    print(f"{name}: groundedness {groundedness:.3f} | floor {GROUNDEDNESS_FLOOR} | "
          f"previous {prev if prev is not None else 'none'} -> {'PASS' if ok else 'FAIL'}")
    if ok:                                  # only a passing run is allowed to move the baseline
        base[name] = groundedness
        json.dump(base, open(BASELINE_PATH, "w"), indent=1)
    return ok


## Cell 6: The row that must stay red
Lesson 4.5 ended on a deliberate failure: at a budget that fits **two of the five** reranked chunks (`small_budget`, stated in chunks because the lane's chunks are ~500-token windows and the loader's clauses ~80 tokens) the packer drops the rest of the probation-and-encashment evidence, so a clause and its citation can go missing.

It would be natural to treat that as a bug. It is more useful as a permanent test. Every other assertion here passes today, which tells you nothing about whether the gate can catch a real regression — you have never watched it catch one. This row is a known, reproducible regression: if the gate stops flagging it, the gate is broken. A smoke detector with a test button.

> If it ever passes, look at which chunk was dropped before you believe the packer improved: the two that fit may simply have held both clauses.

In [ ]:
# Lesson 4.5 ended on a deliberate failure: at a budget that fits two of the five reranked chunks,
# the packer drops the rest and the answer can lose a clause and its citation. That is not a bug
# to fix - it is the row that proves the gate can see a real quality regression. It must stay RED.
RED_ROW = {"id": "budget-small", "shape": "join",
           "question": "If I resign during probation, what notice applies and can I encash leave?",
           "must_contain": ["15", "cannot"], "must_retrieve": ["PB-02", "LV-07"],
           "answerable": True,
           "note": "Run this one with chunk_budget='small' (lesson 4.5's small_budget: two of the five "
                   "chunks fit). It SHOULD fail: the packer cannot fit every clause, so one is missing. "
                   "A gate that passes this row is a gate that would not have caught the regression."}

def check_red_row(run_small) -> bool:
    """run_small: a callable (question, tenant) -> (RAGAnswer, packed chunks) at the small budget.

    Two facts, kept apart (12 September 2026). The DETERMINISTIC one is the packer's: at the small budget a chunk
    holding one of the two clauses is dropped, and that is asserted - it depends on the reranker's order, not on
    the model. The model's answer is printed beside it: an answer that still names both figures from a context
    holding one is not an improvement, it is a claim without evidence, and the row says so."""
    answer, packed = run_small(RED_ROW["question"], TENANT)
    shown = "\n".join(c["chunk_id"] + " " + c["text"] for c in packed)
    dropped = [a for a in RED_ROW["must_retrieve"] if a not in shown]
    text = normalise(answer.answer)
    passed = all(normalise(s) in text for s in RED_ROW["must_contain"])
    print(f"red row: packed {len(packed)} chunk(s); anchors missing from the context: {dropped or 'none'}; "
          f"answer names both figures = {passed}"
          + ("  <- the model answered a clause it was not shown: read the answer, that is not an improvement" if passed and dropped else ""))
    assert dropped, "the small budget kept both clauses: the reranker's order or the packer changed - look at which chunks fit before believing an improvement"
    return not passed

red_is_red = check_red_row(lambda q, t: path_packed(q, t, chunk_budget="small"))
print("red row is red:", red_is_red)


## Cell 7: The comparison — vector, packed, graph
What you are looking for is not a winner but **which path wins on which shape**. Packed should show its advantage on `join` rows, where budget and ordering decide whether both clauses survive. Graph should show its advantage on multi-hop questions and route almost everything else to vector retrieval — if it does not, its router is too eager.

> Be prepared for the sophisticated path not to win. On a corpus of short, well-separated policy clauses, plain dense retrieval can match reranking and traversal. That is a real finding worth more than a technique: it tells you where to stop spending. You cannot predict it from the architecture.

In [ ]:
# Where Module 4 got to, measured rather than asserted.
summary = compare(FRAMES)                    # the paths that ran; a skipped path is absent, not zero
print(summary.to_string())
print("\nRead it this way:")
print("  recall@5 / mrr  -> did retrieval put the required chunks in front of the model?")
print("  answer_match    -> did the answer say what the contract requires?")
print("  refusal_rate    -> did it decline when the corpus has no answer? (over-answering is worse)")
print("  isolation_clean -> False on any row is a release blocker, not a score")

### The gate as a one-liner
One command, a non-zero exit on a failing row. Here it is `promptfoo eval`; on the lane it is `python deploy/evals/run_eval.py --api-url …`, which 4.8 runs against the deployed API and 12.7 runs on every push.

In [ ]:
# The gate as a one-liner, two ways. Both exit non-zero on a failing row, which is all CI needs
# to stop a release:
#   this notebook's runner:   promptfoo eval -c evals/promptfooconfig.yaml --no-progress-bar
#   the lane's gate (4.8):    python deploy/evals/run_eval.py --api-url https://documind-api-NUMBER.us-central1.run.app
# 12.7 runs the second in CI over the committed deploy/evals/golden.jsonl.
print("files this lesson produced (the committed twin is deploy/evals/):")
for f in sorted(os.listdir(EVAL_DIR)):
    print(f"  {f:26} {os.path.getsize(os.path.join(EVAL_DIR, f)):>7,} bytes")


## ✅ Lesson 4.7 complete — and Module 4 with it

- ✅ Wrote a golden set as a contract, with four shapes and an isolation row that blocks a release
- ✅ Computed recall@k and MRR with no judge and no cost, and used them to separate retrieval failure from generation failure
- ✅ Ran a model-graded faithfulness gate against the context each path actually retrieved
- ✅ Pinned the judge, and can say what breaks when it is not pinned
- ✅ Held a floor **and** a moving baseline, so a slow decline fails early
- ✅ Kept one row deliberately red, so the gate is known to work
- ✅ Compared vector and packed retrieval on one corpus, from real runs; the graph path is skipped here unless 4.6's cells are loaded - absent, not zero

**Module 4, in order:** 4.1 ingestion → 4.2 DIY pipeline → 4.3 RAG Engine → 4.4 Search and grounding → 4.5 context engineering → 4.6 graph RAG → 4.7 the gate that decides which of them you ship.

**Next: 4.8 runs this gate against the live lane** — six runs, four service defects, one wrong golden row — then Module 5 — BigQuery ML. Module 10 turns this one-off comparison into continuous evaluation, and 12.7 runs the same gate in CI with an alert for when the baseline slips.